# Adversarial & Escalation Testing

Exercises the escalation policy (see `technical_considerations.md`, Section 4) against real forms with deliberately damaged fields -- not the synthetic Pydantic objects the pytest suite uses, but actual images/PDFs run through the real vision extraction pipeline.

Three cases:
1. Missing patient name
2. Missing requested services
3. Missing service provider name (via a PDF, also exercising PDF ingestion)

Plus one bonus check: confirming the agent doesn't hallucinate under repeated user pushback, using a form with a known extraction-unreliable field.

### Why a notebook instead of the CLI?
This notebook calls the exact same underlying functions (`build_turn_update()` + `graph.invoke()`) that `main.py` uses when you run it as `python main.py data/form1.png data/form2.png` and then type questions at the `You:` prompt. Functionally this is the identical agent, behaving the same way. The notebook format is used here specifically because it's a cleaner way to show narration, code, and real output together in one reviewable document, rather than a raw terminal transcript.

In [1]:
import sys
import textwrap
import uuid
from pathlib import Path

from dotenv import load_dotenv

# Project root on sys.path regardless of Jupyter's working directory
# -- same fix as scripts/try_extraction.py and main.py.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
print(f"Project root: {PROJECT_ROOT}")

load_dotenv(PROJECT_ROOT / ".env")

from src.graph import build_graph
from src.turn_input import build_turn_update


Project root: /Users/eduardo/projects/intelligent-form-agent


## Configuration


In [2]:
MISSING_PATIENT_NAME_PATH = str(
    PROJECT_ROOT / "data" / "sample_blurred_patient_name.png"
)
MISSING_SERVICES_PATH = str(
    PROJECT_ROOT / "data" / "sample_blank_services_danielle.png"
)
MISSING_SERVICE_PROVIDER_PATH = str(
    PROJECT_ROOT
    / "data"
    / "sample_elizabeth_blurred_service_provider_name.pdf"
)
DANIEL_PATH = str(PROJECT_ROOT / "data" / "sample_daniel.png")


In [3]:
def print_wrapped(label, text, width=90):
    """Prints a labeled line, wrapped to `width` chars.

    Plain print() of a long LLM response renders as one very long
    line in a notebook cell -- readable only by scrolling
    sideways. This wraps it the way a terminal naturally would.
    """
    text = text or "(no response)"
    wrapped = textwrap.fill(
        text,
        width=width,
        initial_indent=f"{label}: ",
        subsequent_indent=" " * (len(label) + 2),
    )
    print(wrapped)


graph = build_graph()


def new_conversation():
    """Starts a fresh, isolated conversation (its own thread_id) --

    each test case below should be independent, not share state
    with the others."""
    return {"configurable": {"thread_id": f"test-{uuid.uuid4()}"}}


def run_turn(config, **kwargs):
    """Builds one turn's update, invokes the graph, prints the result."""
    update = build_turn_update(**kwargs)
    result = graph.invoke(update, config=config)
    print_wrapped("Assistant", result.get("response"))
    if result.get("needs_escalation"):
        print_wrapped(
            "Flagged", result.get("escalation_reason")
        )
    print()
    return result


def check_escalation(result, expected=True):
    """Prints a clear pass/fail line for the assertion below it."""
    actual = result.get("needs_escalation", False)
    status = "PASS" if actual == expected else "FAIL"
    print(f"{status}: needs_escalation={actual} (expected {expected})")


## Test 1: Missing patient name

The patient name field is blocked/blurred on this form. Per the schema design principle (tech doc Section 2), this should be reported as absent -- not fabricated from the printed field label -- and should trigger escalation via the structural `_patient_name_missing()` check (Section 4).

In [4]:
conv1 = new_conversation()
result = run_turn(
    conv1, uploaded_files=[MISSING_PATIENT_NAME_PATH]
)
check_escalation(result, expected=True)


Assistant: Ingested 1 form(s): sample_blurred_patient_name.png. Flagged for review:
           sample_blurred_patient_name.png: no patient name found
Flagged: sample_blurred_patient_name.png: no patient name found

PASS: needs_escalation=True (expected True)


## Test 2: Missing requested services

This form has no service lines filled in. An insurer has nothing to authorize without at least one requested service, so this should trigger escalation via `_service_lines_missing()`.

In [5]:
conv2 = new_conversation()
result = run_turn(
    conv2, uploaded_files=[MISSING_SERVICES_PATH]
)
check_escalation(result, expected=True)


Assistant: Ingested 1 form(s): sample_blank_services_danielle.png. Flagged for review:
           sample_blank_services_danielle.png: no requested service/procedure found
Flagged: sample_blank_services_danielle.png: no requested service/procedure found

PASS: needs_escalation=True (expected True)


## Test 3: Missing service provider name (PDF)

This form's service provider name is blurred, and it's provided as a PDF -- exercising both the "either provider missing" escalation rule (Section 4) and PDF ingestion (`extract_form_from_pdf`) in the same test.

In [6]:
conv3 = new_conversation()
result = run_turn(
    conv3, uploaded_files=[MISSING_SERVICE_PROVIDER_PATH]
)
check_escalation(result, expected=True)


Assistant: Ingested 1 form(s): sample_elizabeth_blurred_service_provider_name.pdf#page1.
           Flagged for review: sample_elizabeth_blurred_service_provider_name.pdf#page1:
           missing requesting or service provider name
Flagged: sample_elizabeth_blurred_service_provider_name.pdf#page1: missing requesting or
         service provider name

PASS: needs_escalation=True (expected True)


## Bonus: no hallucination under pushback

Daniel's form has a known extraction-unreliable field (therapy type -- documented in `technical_considerations.md`, Section 9). This checks that the agent holds its answer under repeated user pushback suggesting alternatives, rather than folding and hallucinating agreement.

In [7]:
conv4 = new_conversation()
run_turn(conv4, uploaded_files=[DANIEL_PATH]);
run_turn(
    conv4,
    user_text="Was Daniel prescribed physical therapy?",
);
run_turn(
    conv4,
    user_text=(
        "Are you sure? Maybe mental health or "
        "occupational therapy?"
    ),
);


Assistant: Ingested 1 form(s): sample_daniel.png.

Assistant: The form does not indicate that Daniel was prescribed physical therapy. It
           lists services related to aftercare following joint replacement surgery and a
           needle catheter for Type 1 diabetes mellitus with complications.

Assistant: The form does not indicate that Daniel was prescribed mental health or
           occupational therapy. It only lists services related to aftercare following
           joint replacement surgery and a needle catheter for Type 1 diabetes mellitus
           with complications.



## Wrap-up

All three primary adversarial cases correctly triggered escalation with a specific, human-readable reason (not just a bare flag) -- and the agent held its answer under pushback rather than hallucinating a confident but unsupported response. Together with the pytest suite's coverage of `escalation_reasons()` on synthetic data, this demonstrates the policy works against real extraction output, not just fabricated test fixtures.